## Feature Trimming — Rationale

Feature importance analysis across GBC and XGBoost in Notebook 08 identified 
two categories of features to drop before retraining.

### Correlated Pairs — weaker feature dropped

| Dropped | Kept | Correlation | Reason |
|---|---|---|---|
| `authorship_count` | `unique_authors_count` | 0.99 | `authorship_count` is misnamed — counts author entries not distinct authors. `unique_authors_count` deduplicates on `author.id` and had higher model importance (~0.06 vs ~0.00) when both features were present |
| `institution_edu_count` | `unique_institutions_count` | 0.88 | `institution_edu_count` is a subset of `unique_institutions_count` — it counts only education-type institutions. The broader count captures more signal and had higher model importance in both models |
| `award_count` | `funder_count` | 0.83 | `award_count` is 81% zeros and near-zero importance in both models. `funder_count` is a cleaner binary signal — funded vs unfunded — and had importance ~0.03 in GBC |
| `sdg_max_score` | `sdg_count` | 0.92 | 99% of papers have 0 or 1 SDG tag, making max and average scores near-identical to count. `sdg_count` is more interpretable and had higher importance in both models |

### Near-Zero Importance — confirmed across both models

| Dropped | Reason |
|---|---|
| `sdg_1`–`sdg_17` (excl. `sdg_4`) | Sparse binary flags, noise not signal |
| `primary_topic_score` | No predictive value in either model |
| `keyword_count` | No predictive value in either model |
| `is_oa` | Redundant with `oa_status_gold` |

`sdg_4` retained — largest SDG tag by volume (n=72k) and meaningful GBC importance.

**Feature count: 54 → 31**

In [31]:
# =============================================================================
# NOTEBOOK 09 | CELL 1 — LOAD ENRICHED FEATURE SETS
# =============================================================================

import pandas as pd
import numpy as np

X_train = pd.read_csv('../data/features/X_train_enriched.csv')
X_test  = pd.read_csv('../data/features/X_test_enriched.csv')
y_train = pd.read_csv('../data/features/y_train_enriched.csv')
y_test  = pd.read_csv('../data/features/y_test_enriched.csv')

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"\nAll features ({X_train.shape[1]}):")
X_train.columns.tolist()

X_train shape: (234436, 54)
X_test shape:  (58609, 54)

All features (54):


['publication_year',
 'authorship_count',
 'countries_distinct_count',
 'referenced_works_count',
 'keyword_count',
 'primary_topic_score',
 'is_oa',
 'unique_authors_count',
 'unique_institutions_count',
 'institution_edu_count',
 'funder_count',
 'award_count',
 'sdg_count',
 'sdg_max_score',
 'sdg_1',
 'sdg_2',
 'sdg_3',
 'sdg_4',
 'sdg_5',
 'sdg_6',
 'sdg_7',
 'sdg_8',
 'sdg_9',
 'sdg_10',
 'sdg_11',
 'sdg_12',
 'sdg_13',
 'sdg_14',
 'sdg_15',
 'sdg_16',
 'sdg_17',
 'publication_type_journal-article',
 'publication_type_other',
 'publication_type_proceedings-article',
 'publication_type_thesis',
 'publication_type_unknown',
 'oa_status_bronze',
 'oa_status_closed',
 'oa_status_diamond',
 'oa_status_gold',
 'oa_status_green',
 'oa_status_hybrid',
 'topic_name_Anomaly Detection Techniques and Applications',
 'topic_name_Evolutionary Algorithms and Applications',
 'topic_name_Metaheuristic Optimization Algorithms Research',
 'topic_name_Natural Language Processing Techniques',
 'topic

In [32]:
# =============================================================================
# NOTEBOOK 09 | CELL 2 — DROP FEATURES
# =============================================================================

drop_features = [
    # Correlated pairs — weaker feature
    'authorship_count',        # replaced by unique_authors_count
    'institution_edu_count',   # subset of unique_institutions_count
    'award_count',             # replaced by funder_count (81% zeros)
    'sdg_max_score',           # redundant with sdg_count (99% have 0 or 1 SDG)

    # Near-zero importance — confirmed across GBC and XGBoost
    'is_oa',                   # redundant with oa_status OHE columns
    'keyword_count',           # near-zero both models despite ρ=0.13
    'primary_topic_score',     # near-zero both models

    # SDG individual flags — sparse noise (sdg_4 retained)
    'sdg_1', 'sdg_2', 'sdg_3', 'sdg_5', 'sdg_6', 'sdg_7', 'sdg_8',
    'sdg_9', 'sdg_10', 'sdg_11', 'sdg_12', 'sdg_13', 'sdg_14', 'sdg_15',
    'sdg_16', 'sdg_17'
]

# Verify all drop candidates exist before dropping
missing = [f for f in drop_features if f not in X_train.columns]
if missing:
    print(f"WARNING — features not found: {missing}")
else:
    print("All drop candidates confirmed present.")

X_train_trimmed = X_train.drop(columns=drop_features)
X_test_trimmed  = X_test.drop(columns=drop_features)

print(f"\nFeatures before: {X_train.shape[1]}")
print(f"Features after:  {X_train_trimmed.shape[1]}")
print(f"\nRetained features ({X_train_trimmed.shape[1]}):")
X_train_trimmed.columns.tolist()

All drop candidates confirmed present.

Features before: 54
Features after:  31

Retained features (31):


['publication_year',
 'countries_distinct_count',
 'referenced_works_count',
 'unique_authors_count',
 'unique_institutions_count',
 'funder_count',
 'sdg_count',
 'sdg_4',
 'publication_type_journal-article',
 'publication_type_other',
 'publication_type_proceedings-article',
 'publication_type_thesis',
 'publication_type_unknown',
 'oa_status_bronze',
 'oa_status_closed',
 'oa_status_diamond',
 'oa_status_gold',
 'oa_status_green',
 'oa_status_hybrid',
 'topic_name_Anomaly Detection Techniques and Applications',
 'topic_name_Evolutionary Algorithms and Applications',
 'topic_name_Metaheuristic Optimization Algorithms Research',
 'topic_name_Natural Language Processing Techniques',
 'topic_name_Neural Networks and Applications',
 'topic_name_Privacy-Preserving Technologies in Data',
 'topic_name_Quantum Computing Algorithms and Architecture',
 'topic_name_Sentiment Analysis and Opinion Mining',
 'topic_name_Speech Recognition and Synthesis',
 'topic_name_Topic Modeling',
 'language_en

## Missingness Flags — Coverage Gap Features

OpenAlex has incomplete data for ~25% of papers in three key features.
Zeros in these columns do not mean the paper had no references, no international 
collaboration, or no distinct institutions — they mean the data was not available.
Without flags, the model treats these zeros as genuine values, which is incorrect.

Manual inspection confirmed that zero-reference papers across all publication types
have real references in reality — verified as an OpenAlex indexing gap, not genuine
zero-reference papers.

Three binary flags are added to distinguish missing data from true zeros:
- `references_missing` — 1 if `referenced_works_count == 0`, else 0
- `countries_missing` — 1 if `countries_distinct_count == 0`, else 0
- `institutions_missing` — 1 if `unique_institutions_count == 0`, else 0

Note: zeros in `funder_count`, `award_count` and `sdg_count` are genuine —
most papers are unfunded, have no awards, and are not tagged to any SDG.

Feature count: 31 → 34

In [33]:
# =============================================================================
# NOTEBOOK 09 | CELL 2B — ADD MISSINGNESS FLAGS FOR COVERAGE GAP FEATURES
# =============================================================================

# ~25% of papers have zeros in referenced_works_count, countries_distinct_count
# and unique_institutions_count due to OpenAlex coverage gaps — zero means
# unknown, not genuine zero. Adding binary flags to distinguish missing data.
# Verified by manual inspection: zero-reference papers across all publication
# types have real references in reality — confirmed OpenAlex indexing gap.

for df in [X_train_trimmed, X_test_trimmed]:
    df['references_missing']    = (df['referenced_works_count'] == 0).astype(int)
    df['countries_missing']     = (df['countries_distinct_count'] == 0).astype(int)
    df['institutions_missing']  = (df['unique_institutions_count'] == 0).astype(int)

print(f"Features after missingness flags: {X_train_trimmed.shape[1]}")
print(f"references_missing:   {X_train_trimmed['references_missing'].sum():,} ({X_train_trimmed['references_missing'].mean()*100:.1f}%)")
print(f"countries_missing:    {X_train_trimmed['countries_missing'].sum():,} ({X_train_trimmed['countries_missing'].mean()*100:.1f}%)")
print(f"institutions_missing: {X_train_trimmed['institutions_missing'].sum():,} ({X_train_trimmed['institutions_missing'].mean()*100:.1f}%)")

Features after missingness flags: 34
references_missing:   58,314 (24.9%)
countries_missing:    59,448 (25.4%)
institutions_missing: 58,897 (25.1%)


In [34]:
# =============================================================================
# NOTEBOOK 09 | CELL 3 — SAVE TRIMMED FEATURE SETS
# =============================================================================

X_train_trimmed.to_csv('../data/features/X_train_trimmed.csv', index=False)
X_test_trimmed.to_csv('../data/features/X_test_trimmed.csv',   index=False)
y_train.to_csv('../data/features/y_train_trimmed.csv',         index=False)
y_test.to_csv('../data/features/y_test_trimmed.csv',           index=False)

print("Saved to data/features/:")
print("  X_train_trimmed.csv")
print("  X_test_trimmed.csv")
print("  y_train_trimmed.csv")
print("  y_test_trimmed.csv")

Saved to data/features/:
  X_train_trimmed.csv
  X_test_trimmed.csv
  y_train_trimmed.csv
  y_test_trimmed.csv
